# TC 5033
## Advanced Machine Learning Methods
## Transformers

### Team 30
- A01796272 - Luis Antonio Ramirez Martinez
- A01796323 - Benjamin Cisneros Barraza
- A01796363 - Arthur Jafed Zizumbo Velasco
- A01796937 - Sandra Luz Cervantes Espinoza

#### Activity 4: Implementing a Translator

- Objective

To understand the Transformer Architecture by Implementing a translator.

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in one of [week's 9 videos](https://youtu.be/XefFj4rLHgU)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  You should test your translator with at least 10 sentences. The dataset used for this task was obtained from [Tatoeba, a large dataset of sentences and translations](https://tatoeba.org/en/downloads).
  
- Evaluation Criteria

    - Code Readability and Comments
    - Traning a translator
    - Translating at least 10 sentences.

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.



## Imports

This notebook uses the following libraries:

| Library | Purpose |
|---|---|
| `pandas` | Load and manipulate the dataset (CSV / TSV files) |
| `torch` / `torch.nn` | PyTorch framework for building and training the Transformer |
| `torch.optim` | Adam optimizer for gradient-based training |
| `torch.utils.data` | `Dataset` and `DataLoader` utilities for batching |
| `collections.Counter` | Count word frequencies when building vocabularies |
| `math` / `numpy` | Mathematical operations (square root, logarithm, etc.) |
| `re` | Regular expressions for text preprocessing |
| `gc` | Garbage collection — used to free GPU/MPS memory |

In [1]:
import pandas as pd
from google.colab import drive
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re
import gc

#### Script to convert csv to text file

In [2]:
#This script requires to convert the TSV file to CSV
# easiest way is to open it in Calc or excel and save as csv
drive.mount('/content/drive', force_remount=True)
PATH = '/content/drive/MyDrive/Colab Notebooks/MNA/Advance Machine Learning/A4/data/en_esp.csv'

df = pd.read_csv(PATH)

Mounted at /content/drive


In [63]:
eng_spa_cols = df.iloc[:, [1, 3]]
eng_spa_cols['length'] = eng_spa_cols.iloc[:, 0].str.len()
eng_spa_cols = eng_spa_cols.sort_values(by='length')
eng_spa_cols = eng_spa_cols.drop(columns=['length'])

output_file_path = '/content/drive/MyDrive/Colab Notebooks/MNA/Advance Machine Learning/A4/data/eng-spa4.txt'
eng_spa_cols.to_csv(output_file_path, sep='\t', index=False, header=False)

/tmp/ipykernel_1164/2868161458.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eng_spa_cols['length'] = eng_spa_cols.iloc[:, 0].str.len()


### Dataset Preparation

The raw dataset is loaded from a CSV file containing English–Spanish sentence pairs sourced from [Tatoeba](https://tatoeba.org/en/downloads).

The two relevant columns (English and Spanish) are extracted and then **sorted by English sentence length**. Sorting by length is a common trick that groups similar-length sequences into the same batch, which reduces the amount of padding needed and makes training more efficient. The result is saved as a tab-separated `.txt` file for later use.

## Transformer - Attention is all you need

### Architecture Overview

The Transformer model, introduced in the paper *"Attention Is All You Need"* (Vaswani et al., 2017), replaces recurrent networks (RNNs/LSTMs) with a fully attention-based design. This makes it faster to train (highly parallelizable) and better at capturing long-range dependencies.

The model has two main parts:
- **Encoder** — reads the entire source sequence and builds a rich contextual representation of it.
- **Decoder** — generates the output sequence one token at a time, attending to both its own previous outputs and the encoder's output.

Both are stacks of identical layers that combine **Multi-Head Self-Attention** and a small **Feed-Forward Network**, with residual connections and layer normalization after each step.

```
Input (English)                          Output (Spanish)
     │                                         │
  [Embedding + Positional Encoding]     [Embedding + Positional Encoding]
     │                                         │
  ┌──┴──────────────────┐              ┌───────┴──────────────────────┐
  │      Encoder        │              │          Decoder             │
  │  (N identical       │──encoder──►  │  (N identical layers, each   │
  │   layers)           │   output     │   with masked self-attention │
  └─────────────────────┘              │   + cross-attention)         │
                                       └──────────────────────────────┘
                                                     │
                                            Linear + Softmax
                                                     │
                                           Predicted next word
```

**Key components defined in the cells below:**

| Component | Role |
|---|---|
| `PositionalEmbedding` | Adds position information to token embeddings |
| `MultiHeadAttention` | Computes attention in parallel across multiple subspaces |
| `PositionFeedForward` | Two-layer MLP applied independently to each position |
| `EncoderSubLayer` | One encoder block: self-attention + feed-forward + residuals |
| `Encoder` | Stack of N encoder sub-layers |
| `DecoderSubLayer` | One decoder block: masked self-attention + cross-attention + feed-forward |
| `Decoder` | Stack of N decoder sub-layers |

In [4]:
torch.manual_seed(23)

In [29]:
# -----------------------------------------------------------------------------
# Device Selection: CUDA (NVIDIA GPU), MPS (Apple Silicon), or CPU
# -----------------------------------------------------------------------------
# This block determines the best available hardware accelerator for PyTorch.
# Priority:
#   1) CUDA  - If an NVIDIA GPU is available.
#   2) MPS   - If running on macOS with Apple Silicon (M1/M2/M3) and Metal backend.
#   3) CPU   - Fallback option when no GPU acceleration is available.
# -----------------------------------------------------------------------------

# Check if CUDA is available (NVIDIA GPUs)
if torch.cuda.is_available():
    device = torch.device('cuda')

# If not, check for Apple's Metal Performance Shaders (MPS) backend
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device('mps')
    gc.collect()
    torch.mps.empty_cache()

# If neither CUDA nor MPS is available, default to CPU
else:
    device = torch.device('cpu')

# Print the selected device for confirmation
print(f"[INFO] Device: {device}")

[INFO] Device: cuda


In [5]:
MAX_SEQ_LEN = 128

### Core Components

---

#### 1. Positional Embedding

Since the Transformer has no recurrence or convolution, it has no built-in sense of word order. `PositionalEmbedding` fixes this by adding a fixed sinusoidal signal to each token's embedding. The encoding uses sine functions for even dimensions and cosine for odd dimensions at different frequencies, so each position gets a unique pattern the model can learn to interpret.

---

#### 2. Multi-Head Attention

Attention lets the model decide **how much to focus on each other token** when processing a given token. The formula is:

`Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V`

- **Q (Query)**: what the current token is looking for
- **K (Key)**: what each token offers
- **V (Value)**: the actual content to retrieve

**Multi-Head** means we run this attention mechanism `num_heads` times in parallel (each on a smaller slice of the embedding), then concatenate the results. This allows the model to jointly attend to information from different representation subspaces.

---

#### 3. Position-wise Feed-Forward Network

After attention, a small two-layer MLP (`Linear → ReLU → Linear`) is applied to each position independently. This adds non-linearity and lets the model further transform each token's representation.

---

#### 4. Encoder Sub-Layer

Each encoder block applies the following steps with residual connections and layer normalization:
1. Self-attention (each token attends to all tokens in the source)
2. Add & Norm
3. Feed-forward network
4. Add & Norm

---

#### 5. Decoder Sub-Layer

Each decoder block applies:
1. **Masked** self-attention — target tokens can only attend to previous positions (no peeking at future tokens)
2. Add & Norm
3. **Cross-attention** — attends to the encoder's output (connects source and target)
4. Add & Norm
5. Feed-forward network
6. Add & Norm

In [53]:
class PositionalEmbedding(nn.Module):
    """Adds sinusoidal positional encoding to token embeddings.

    Since the Transformer has no recurrence, this module injects position
    information by adding a fixed sine/cosine pattern to each token embedding.
    Even dimensions use sin, odd dimensions use cos, each at a different frequency.

    Args:
        d_model (int): Embedding dimension.
        max_seq_len (int): Maximum sequence length supported.
    """
    def __init__(self, d_model, max_seq_len = MAX_SEQ_LEN):
        super().__init__()
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        token_pos = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0)/d_model))
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term)
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term)
        self.pos_embed_matrix = self.pos_embed_matrix.unsqueeze(0).transpose(0,1)

    def forward(self, x):
        """Add positional encoding to the input embeddings.

        Args:
            x (Tensor): Token embeddings of shape [seq_len, batch, d_model].

        Returns:
            Tensor: Embeddings with positional encoding added, same shape as x.
        """
#         print(self.pos_embed_matrix.shape)
#         print(x.shape)
        return x + self.pos_embed_matrix[:x.size(0), :]

class MultiHeadAttention(nn.Module):
    """Multi-Head Scaled Dot-Product Attention.

    Splits the embedding into `num_heads` subspaces and computes attention
    independently in each, then concatenates and projects the results.
    This allows the model to attend to information from different representation
    subspaces simultaneously.

    Args:
        d_model (int): Total embedding dimension. Must be divisible by num_heads.
        num_heads (int): Number of parallel attention heads.
    """
    def __init__(self, d_model = 512, num_heads = 8):
        super().__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num heads'

        self.d_v = d_model // num_heads
        self.d_k = self.d_v
        self.num_heads = num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask = None):
        """Compute multi-head attention.

        Args:
            Q (Tensor): Query tensor [batch, seq_len, d_model].
            K (Tensor): Key tensor   [batch, seq_len, d_model].
            V (Tensor): Value tensor [batch, seq_len, d_model].
            mask (Tensor, optional): Boolean mask; positions where mask==0 get
                score -1e9 (effectively ignored after softmax).

        Returns:
            tuple: (weighted_values [batch, seq_len, d_model], attention weights)
        """
        batch_size = Q.size(0)
        '''
        Q, K, V -> [batch_size, seq_len, num_heads*d_k]
        after transpose Q, K, V -> [batch_size, num_heads, seq_len, d_k]
        '''
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )

        weighted_values, attention = self.scale_dot_product(Q, K, V, mask)
        weighted_values = weighted_values.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads*self.d_k)
        weighted_values = self.W_o(weighted_values)

        return weighted_values, attention


    def scale_dot_product(self, Q, K, V, mask = None):
        """Scaled dot-product attention: softmax(QKᵀ / √d_k) · V.

        Args:
            Q, K, V (Tensor): Shape [batch, heads, seq_len, d_k].
            mask (Tensor, optional): Positions to mask out.

        Returns:
            tuple: (weighted_values, attention_weights)
        """
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = F.softmax(scores, dim = -1)
        weighted_values = torch.matmul(attention, V)

        return weighted_values, attention


class PositionFeedForward(nn.Module):
    """Position-wise Feed-Forward Network (FFN).

    Applies a two-layer MLP independently to each position:
        FFN(x) = max(0, xW1 + b1)W2 + b2

    Args:
        d_model (int): Input and output dimension.
        d_ff (int): Hidden dimension (typically 4 * d_model).
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        """Apply FFN: Linear -> ReLU -> Linear.

        Args:
            x (Tensor): Input of shape [batch, seq_len, d_model].

        Returns:
            Tensor: Output of shape [batch, seq_len, d_model].
        """
        return self.linear2(F.relu(self.linear1(x)))

class EncoderSubLayer(nn.Module):
    """Single encoder layer: self-attention + feed-forward with residuals.

    Applies:
        1. Multi-head self-attention
        2. Add & LayerNorm
        3. Position-wise FFN
        4. Add & LayerNorm

    Args:
        d_model (int): Embedding dimension.
        num_heads (int): Number of attention heads.
        d_ff (int): Hidden dimension of the FFN.
        dropout (float): Dropout probability.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.droupout1 = nn.Dropout(dropout)
        self.droupout2 = nn.Dropout(dropout)

    def forward(self, x, mask = None):
        """Forward pass through one encoder layer.

        Args:
            x (Tensor): Input [batch, seq_len, d_model].
            mask (Tensor, optional): Padding mask for the source sequence.

        Returns:
            Tensor: Output [batch, seq_len, d_model].
        """
        attention_score, _ = self.self_attn(x, x, x, mask)
        x = x + self.droupout1(attention_score)
        x = self.norm1(x)
        x = x + self.droupout2(self.ffn(x))
        return self.norm2(x)

class Encoder(nn.Module):
    """Stack of N identical encoder layers followed by a final LayerNorm.

    Args:
        d_model (int): Embedding dimension.
        num_heads (int): Number of attention heads.
        d_ff (int): FFN hidden dimension.
        num_layers (int): Number of encoder sub-layers to stack.
        dropout (float): Dropout probability.
    """

    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderSubLayer(nn.Module):
    """Single decoder layer: masked self-attention + cross-attention + FFN.

    Applies:
        1. Masked multi-head self-attention (causal — no future peeking)
        2. Add & LayerNorm
        3. Cross-attention over encoder output
        4. Add & LayerNorm
        5. Position-wise FFN
        6. Add & LayerNorm

    Args:
        d_model (int): Embedding dimension.
        num_heads (int): Number of attention heads.
        d_ff (int): FFN hidden dimension.
        dropout (float): Dropout probability.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, target_mask=None, encoder_mask=None):
        attention_score, _ = self.self_attn(x, x, x, target_mask)
        x = x + self.dropout1(attention_score)
        x = self.norm1(x)

        encoder_attn, _ = self.cross_attn(x, encoder_output, encoder_output, encoder_mask)
        x = x + self.dropout2(encoder_attn)
        x = self.norm2(x)

        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        return self.norm3(x)

class Decoder(nn.Module):
    """Stack of N identical decoder layers followed by a final LayerNorm.
    Args:
        d_model (int): Embedding dimension.
        num_heads (int): Number of attention heads.
        d_ff (int): FFN hidden dimension.
        num_layers (int): Number of decoder sub-layers to stack.
        dropout (float): Dropout probability.
    """

    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, target_mask, encoder_mask):
        """Pass target through all decoder layers.

        Args:
            x (Tensor): Target embeddings [batch, tgt_len, d_model].
            encoder_output (Tensor): Encoder output [batch, src_len, d_model].
            target_mask (Tensor): Causal mask for target.
            encoder_mask (Tensor): Padding mask for source.

        Returns:
            Tensor: Decoded representation [batch, tgt_len, d_model].
        """
        for layer in self.layers:
            x = layer(x, encoder_output, target_mask, encoder_mask)
        return self.norm(x)

### Full Transformer Model

The `Transformer` class assembles all the components into one end-to-end model:

1. **Embedding layers** — convert token indices to dense vectors (one for the source language, one for the target)
2. **Positional encoding** — adds position information to both embeddings
3. **Encoder** — processes the full source (English) sequence
4. **Decoder** — generates the target (Spanish) sequence, attending to both itself and the encoder output
5. **Output linear layer** — projects each decoder position to a logit vector over the full target vocabulary

The `mask` method creates two masks:
- **Source mask**: hides `<pad>` tokens in the encoder input so they don't influence attention scores
- **Target mask**: combines a padding mask with a **causal (lower-triangular) mask** that prevents the decoder from attending to future tokens — this is critical to avoid "cheating" during training

In [56]:
class Transformer(nn.Module):
    """Full Encoder-Decoder Transformer for sequence-to-sequence tasks.

    Combines token embeddings, positional encoding, an encoder stack, and a
    decoder stack into one end-to-end model. The final linear layer projects
    the decoder output to logits over the target vocabulary.

    Args:
        d_model (int): Embedding and model dimension.
        num_heads (int): Number of attention heads.
        d_ff (int): FFN hidden dimension.
        num_layers (int): Number of encoder and decoder layers.
        input_vocab_size (int): Source vocabulary size.
        target_vocab_size (int): Target vocabulary size.
        max_len (int): Maximum sequence length.
        dropout (float): Dropout probability.
    """
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                 input_vocab_size, target_vocab_size,
                 max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.pos_embedding = PositionalEmbedding(d_model, max_len)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_layer = nn.Linear(d_model, target_vocab_size)

    def forward(self, source, target):
        """Run a full forward pass through the Transformer.

        Args:
            source (Tensor): Source token indices [batch, src_len].
            target (Tensor): Target token indices [batch, tgt_len].

        Returns:
            Tensor: Logits over the target vocabulary [batch, tgt_len, target_vocab_size].
        """
        # Encoder mask
        source_mask, target_mask = self.mask(source, target)
        # Embedding and positional Encoding
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        # Encoder
        encoder_output = self.encoder(source, source_mask)

        # Decoder embedding and postional encoding
        target = self.decoder_embedding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        # Decoder
        output = self.decoder(target, encoder_output, target_mask, source_mask)

        return self.output_layer(output)



    def mask(self, source, target):
        """Create padding and causal masks for source and target sequences.

        The source mask hides padding tokens. The target mask combines a padding
        mask with a lower-triangular causal mask so the decoder cannot attend
        to future positions during training.

        Args:
            source (Tensor): Source token indices [batch, src_len].
            target (Tensor): Target token indices [batch, tgt_len].

        Returns:
            tuple: (source_mask, target_mask)
        """
        source_mask = (source != 0).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0).unsqueeze(1).unsqueeze(2)
        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        target_mask = target_mask & no_mask
        return source_mask, target_mask


#### Simple test

In [8]:
seq_len_source = 10
seq_len_target = 10
batch_size = 2
input_vocab_size = 50
target_vocab_size = 50

source = torch.randint(1, input_vocab_size, (batch_size, seq_len_source))
target = torch.randint(1, target_vocab_size, (batch_size, seq_len_target))

In [35]:
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

model = Transformer(d_model, num_heads, d_ff, num_layers,
                  input_vocab_size, target_vocab_size,
                  max_len=MAX_SEQ_LEN, dropout=0.1)

model = model.to(device)
source = source.to(device)
target = target.to(device)

In [36]:
output = model(source, target)

In [37]:
# Expected output shape -> [batch, seq_len_target, target_vocab_size] i.e. [2, 10, 50]
print(f'ouput.shape {output.shape}')

ouput.shape torch.Size([2, 10, 50])


The output shape `[2, 10, 50]` confirms the model is working correctly:
- `2` → batch size
- `10` → target sequence length
- `50` → target vocabulary size (one logit per possible word)

At each position, the model produces a probability distribution over the entire vocabulary. The predicted word at each step is the one with the highest score.

### Translator Eng-Spa

In [12]:
PATH = '/content/drive/MyDrive/Colab Notebooks/MNA/Advance Machine Learning/A4/data/eng-spa4.txt'

In [13]:
with open(PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()
eng_spa_pairs = [line.strip().split('\t') for line in lines if '\t' in line]

The dataset contains over 100,000 English–Spanish sentence pairs. Each line is tab-separated: the first column is the English sentence and the second is the Spanish translation. A few examples:

In [14]:
eng_spa_pairs[:10]

[['Go.', 'Vayan.'],
 ['Go.', 'Id.'],
 ['Hi.', '¡Hola!'],
 ['Go!', '¡Sal!'],
 ['Go!', '¡Ve!'],
 ['OK.', 'Bueno.'],
 ['Go!', '¡Ya!'],
 ['Go!', '¡Fuera!'],
 ['Go!', '¡Vete!'],
 ['Go!', '¡Váyase!']]

In [15]:
eng_sentences = [pair[0] for pair in eng_spa_pairs]
spa_sentences = [pair[1] for pair in eng_spa_pairs]

In [16]:
print(eng_sentences[:10])
print(spa_sentences[:10])


['Go.', 'Go.', 'Hi.', 'Go!', 'Go!', 'OK.', 'Go!', 'Go!', 'Go!', 'Go!']
['Vayan.', 'Id.', '¡Hola!', '¡Sal!', '¡Ve!', 'Bueno.', '¡Ya!', '¡Fuera!', '¡Vete!', '¡Váyase!']


### Text Preprocessing

Before building the vocabulary, we normalize the text to reduce noise and keep the vocabulary size manageable:

- **Lowercase** — treat "Hello" and "hello" as the same word
- **Remove accents** (á→a, é→e, í→i, ó→o, ú→u) — simplifies the Spanish vocabulary significantly
- **Remove non-alphabetic characters** — strips punctuation, numbers, and special symbols
- **Add `<sos>` and `<eos>` tokens** — these special tokens mark the start and end of each sequence. The decoder uses `<sos>` to know when to begin generating, and stops when it produces `<eos>`

In [57]:
def preprocess_sentence(sentence):
    """Normalize a sentence for use as model input.

    Applies the following steps in order:
        1. Lowercase and strip whitespace.
        2. Collapse multiple spaces.
        3. Remove Spanish accents (á→a, é→e, í→i, ó→o, ú→u).
        4. Remove any character that is not a lowercase letter.
        5. Wrap the result with <sos> and <eos> special tokens.

    Args:
        sentence (str): Raw input sentence in English or Spanish.

    Returns:
        str: Cleaned sentence wrapped with <sos> and <eos>.

    Example:
        >>> preprocess_sentence("¿Hola @ cómo estás? 123")
        '<sos> hola como estas <eos>'
    """
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[á]+", "a", sentence)
    sentence = re.sub(r"[é]+", "e", sentence)
    sentence = re.sub(r"[í]+", "i", sentence)
    sentence = re.sub(r"[ó]+", "o", sentence)
    sentence = re.sub(r"[ú]+", "u", sentence)
    sentence = re.sub(r"[^a-z]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<sos> ' + sentence + ' <eos>'
    return sentence

In [18]:
s1 = '¿Hola @ cómo estás? 123'

In [19]:
print(s1)
print(preprocess_sentence(s1))

¿Hola @ cómo estás? 123
<sos> hola como estas <eos>


In [20]:
eng_sentences = [preprocess_sentence(sentence) for sentence in eng_sentences]
spa_sentences = [preprocess_sentence(sentence) for sentence in spa_sentences]

In [22]:
spa_sentences[:10]

['<sos> vayan <eos>',
 '<sos> id <eos>',
 '<sos> hola <eos>',
 '<sos> sal <eos>',
 '<sos> ve <eos>',
 '<sos> bueno <eos>',
 '<sos> ya <eos>',
 '<sos> fuera <eos>',
 '<sos> vete <eos>',
 '<sos> vayase <eos>']

### Vocabulary Building

We build separate word-to-index vocabularies for English and Spanish. Each unique word gets an integer ID.

- Words are sorted by **frequency** (most common first), so the most frequent words get the lowest indices
- Index `0` → `<pad>` — used to fill shorter sequences so all sequences in a batch have equal length
- Index `1` → `<unk>` — used for words not seen during training (out-of-vocabulary words)
- All other words start at index `2`

Two complementary mappings are created: `word2idx` (word → index) for encoding input, and `idx2word` (index → word) for decoding model output back into text.

In [58]:
def build_vocab(sentences):
    """Build word-to-index and index-to-word mappings from a list of sentences.

    Words are sorted by descending frequency so the most common words get
    the lowest indices. Two special tokens are always included:
        - Index 0: <pad>  — padding token (ignored during training and decoding)
        - Index 1: <unk>  — unknown token (used for words not seen during training)

    Args:
        sentences (list[str]): Preprocessed sentences (words separated by spaces).

    Returns:
        tuple:
            word2idx (dict): Maps each word to a unique integer index.
            idx2word (dict): Reverse mapping from index to word.
    """
    words = [word for sentence in sentences for word in sentence.split()]
    word_count = Counter(words)
    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)
    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_counts, 2)}
    word2idx['<pad>'] = 0
    word2idx['<unk>'] = 1
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

In [22]:
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
spa_word2idx, spa_idx2word = build_vocab(spa_sentences)
eng_vocab_size = len(eng_word2idx)
spa_vocab_size = len(spa_word2idx)

In [23]:
print(eng_vocab_size, spa_vocab_size)

28415 48412


### Dataset and DataLoader

The `EngSpaDataset` class wraps the preprocessed sentence pairs into a PyTorch `Dataset`. When accessed by index, it returns a pair of tensors: the English sentence and the Spanish sentence, both as sequences of integer indices.

The `collate_fn` function is called by the `DataLoader` to assemble individual samples into a batch:
- Each sequence is **truncated** to `MAX_SEQ_LEN` tokens if it exceeds the limit
- Shorter sequences are **padded with zeros** (`<pad>` token) so all sequences in the batch have the same length — this is required for efficient GPU matrix operations

In [59]:
class EngSpaDataset(Dataset):
    """PyTorch Dataset for English-Spanish sentence pairs.

    Converts preprocessed sentence strings to tensors of token indices
    using the provided vocabularies.

    Args:
        eng_sentences (list[str]): Preprocessed English sentences.
        spa_sentences (list[str]): Preprocessed Spanish sentences.
        eng_word2idx (dict): English word-to-index vocabulary.
        spa_word2idx (dict): Spanish word-to-index vocabulary.
    """
    def __init__(self, eng_sentences, spa_sentences, eng_word2idx, spa_word2idx):
        self.eng_sentences = eng_sentences
        self.spa_sentences = spa_sentences
        self.eng_word2idx = eng_word2idx
        self.spa_word2idx = spa_word2idx

    def __len__(self):
        return len(self.eng_sentences)

    def __getitem__(self, idx):
        eng_sentence = self.eng_sentences[idx]
        spa_sentence = self.spa_sentences[idx]
        # return tokens idxs
        eng_idxs = [self.eng_word2idx.get(word, self.eng_word2idx['<unk>']) for word in eng_sentence.split()]
        spa_idxs = [self.spa_word2idx.get(word, self.spa_word2idx['<unk>']) for word in spa_sentence.split()]

        return torch.tensor(eng_idxs), torch.tensor(spa_idxs)

In [60]:
def collate_fn(batch):
    """Collate a list of (eng, spa) tensor pairs into padded batch tensors.

    Each sequence is first truncated to MAX_SEQ_LEN, then all sequences in
    the batch are padded with 0 (<pad>) to match the length of the longest
    sequence in the batch.

    Args:
        batch (list[tuple]): List of (eng_tensor, spa_tensor) pairs from the Dataset.

    Returns:
        tuple:
            eng_batch (Tensor): Padded English batch [batch_size, max_eng_len].
            spa_batch (Tensor): Padded Spanish batch [batch_size, max_spa_len].
    """
    eng_batch, spa_batch = zip(*batch)
    eng_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in eng_batch]
    spa_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spa_batch]
    eng_batch = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=0)
    spa_batch = torch.nn.utils.rnn.pad_sequence(spa_batch, batch_first=True, padding_value=0)
    return eng_batch, spa_batch


### Training Loop

The `train` function runs the standard supervised learning procedure:

1. **Teacher forcing** — the Spanish sequence is split into:
   - `target_input`: all tokens except the last (fed to the decoder as context)
   - `target_output`: all tokens except the first (the ground truth to predict)
   
   This means the decoder always receives the *correct* previous token during training, which leads to much faster and more stable convergence compared to feeding back the model's own predictions.

2. The model predicts a probability distribution over the Spanish vocabulary at every position in the target sequence.

3. **Cross-entropy loss** is computed between the predicted distributions and the true next tokens. Padding positions are excluded from the loss (`ignore_index=0`).

4. **Backpropagation** computes gradients and the **Adam optimizer** updates the model weights.

In [61]:
def train(model, dataloader, loss_function, optimiser, epochs):
    """Train the Transformer model using teacher forcing.

    For each batch, the Spanish sequence is split into:
        - target_input:  all tokens except the last  (fed to the decoder)
        - target_output: all tokens except the first (ground truth labels)

    This is called teacher forcing: the decoder always receives the correct
    previous token during training, which speeds up convergence.

    Loss is computed with CrossEntropyLoss, ignoring padding tokens (index 0).
    Gradients are computed via backpropagation and weights are updated with Adam.

    Args:
        model (nn.Module): The Transformer model to train.
        dataloader (DataLoader): Provides batches of (eng, spa) tensor pairs.
        loss_function (nn.Module): Loss criterion (e.g. CrossEntropyLoss).
        optimiser (Optimizer): Optimizer (e.g. Adam).
        epochs (int): Number of full passes over the dataset.
    """
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for i, (eng_batch, spa_batch) in enumerate(dataloader):
            eng_batch = eng_batch.to(device)
            spa_batch = spa_batch.to(device)
            # Decoder preprocessing
            target_input = spa_batch[:, :-1]
            target_output = spa_batch[:, 1:].contiguous().view(-1)
            # Zero grads
            optimiser.zero_grad()
            # run model
            output = model(eng_batch, target_input)
            output = output.view(-1, output.size(-1))
            # loss\
            loss = loss_function(output, target_output)
            # gradient and update parameters
            loss.backward()
            optimiser.step()
            total_loss += loss.item()

        avg_loss = total_loss/len(dataloader)
        print(f'Epoch: {epoch}/{epochs}, Loss: {avg_loss:.4f}')



In [27]:
BATCH_SIZE = 64
dataset = EngSpaDataset(eng_sentences, spa_sentences, eng_word2idx, spa_word2idx)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [30]:
# Original
model = Transformer(d_model=512, num_heads=8, d_ff=2048, num_layers=6,
                    input_vocab_size=eng_vocab_size, target_vocab_size=spa_vocab_size,
                    max_len=MAX_SEQ_LEN, dropout=0.1)

### Hyperparameters and Training Setup

The model uses the original hyperparameters from the *"Attention Is All You Need"* paper:

| Parameter | Value | Description |
|---|---|---|
| `d_model` | 512 | Embedding size and all internal representation dimensions |
| `num_heads` | 8 | Number of attention heads (each head has `d_k = 512/8 = 64`) |
| `d_ff` | 2048 | Hidden dimension of the feed-forward network |
| `num_layers` | 6 | Number of encoder and decoder layers stacked |
| `dropout` | 0.1 | Dropout probability for regularization |
| `BATCH_SIZE` | 16 | Sentence pairs per training step |
| `lr` | 0.0001 | Learning rate for the Adam optimizer |
| `epochs` | 10 | Full passes over the training data |

> **Note:** A smaller version (`d_model=128`, 2 layers) is available (commented out) for machines with limited GPU/MPS memory.

In [31]:
model = model.to(device)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimiser = optim.Adam(model.parameters(), lr=0.0001)


In [32]:
train(model, dataloader, loss_function, optimiser, epochs = 10)

Epoch: 0/10, Loss: 3.5656
Epoch: 1/10, Loss: 2.1740
Epoch: 2/10, Loss: 1.6810
Epoch: 3/10, Loss: 1.3581
Epoch: 4/10, Loss: 1.1137
Epoch: 5/10, Loss: 0.9156
Epoch: 6/10, Loss: 0.7549
Epoch: 7/10, Loss: 0.6299
Epoch: 8/10, Loss: 0.5367
Epoch: 9/10, Loss: 0.4702


### Inference — Translating New Sentences

After training, we use **greedy decoding** to translate new sentences:

1. The input English sentence is preprocessed and converted to a sequence of indices
2. The encoder processes the full source sequence once and produces the encoder output
3. The decoder starts with the `<sos>` token and generates one new token per step
4. At each step, the token with the **highest logit (argmax)** is selected as the next word
5. Generation stops when the `<eos>` token is produced or `max_len` is reached

This approach is called *greedy* decoding because we always pick the single best token at each step. A more powerful alternative is **beam search**, which keeps track of the top-K candidate sequences at every step and often produces better translations — but greedy decoding is simpler and fast enough for demonstration purposes.

In [62]:
def sentence_to_indices(sentence, word2idx):
    """Convert a whitespace-tokenized sentence string to a list of token indices.

    Words not found in the vocabulary are mapped to the <unk> index.

    Args:
        sentence (str): Preprocessed sentence (words separated by spaces).
        word2idx (dict): Vocabulary mapping word -> index.

    Returns:
        list[int]: Sequence of token indices.
    """
    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

def indices_to_sentence(indices, idx2word):
    """Convert a list of token indices back to a readable sentence string.

    Skips padding tokens (<pad>) from the output.

    Args:
        indices (list[int]): Sequence of token indices.
        idx2word (dict): Vocabulary mapping index -> word.

    Returns:
        str: Decoded sentence (space-separated words).
    """
    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

def translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    """Translate a single English sentence to Spanish using greedy decoding.

    The encoder processes the full source sequence once. The decoder then
    generates one token at a time: at each step the token with the highest
    logit (argmax) is appended to the output and fed back as input for the
    next step. Generation stops at <eos> or when max_len is reached.

    Args:
        model (nn.Module): Trained Transformer model.
        sentence (str): Raw English input sentence.
        eng_word2idx (dict): English word-to-index vocabulary.
        spa_idx2word (dict): Spanish index-to-word vocabulary.
        max_len (int): Maximum number of tokens to generate.
        device (str or torch.device): Device to run inference on.

    Returns:
        str: Translated Spanish sentence.
    """
    model.eval()
    sentence = preprocess_sentence(sentence)
    input_indices = sentence_to_indices(sentence, eng_word2idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Initialize the target tensor with <sos> token
    tgt_indices = [spa_word2idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == spa_word2idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, spa_idx2word)

In [34]:
def evaluate_translations(model, sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    for sentence in sentences:
        translation = translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len, device)
        print(f'Input sentence: {sentence}')
        print(f'Traducción: {translation}')
        print()

# Example sentences to test the translator
test_sentences = [
    # Greetings and simple phrases
    "Hello, how are you?",
    "Good morning!",
    "Good night!",
    # Daily life
    "I am hungry.",
    "I want to eat.",
    "Where is the bathroom?",
    "What time is it?",
    # Learning and technology
    "I am learning artificial intelligence.",
    "Artificial intelligence is great.",
    "The computer is on the table.",
    # Slightly longer sentences
    "She reads a book every day.",
    "We are students at the university.",
    "The dog runs in the park.",
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Evaluate translations
evaluate_translations(model, test_sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)


Input sentence: Hello, how are you?
Traducción: <sos> hola que tal <eos>

Input sentence: Good morning!
Traducción: <sos> buenos dias <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

Input sentence: I am hungry.
Traducción: <sos> estoy hambriento <eos>

Input sentence: I want to eat.
Traducción: <sos> quiero comer <eos>

Input sentence: Where is the bathroom?
Traducción: <sos> donde esta el ba o <eos>

Input sentence: What time is it?
Traducción: <sos> que hora es <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> estoy aprendiendo inteligencia artificial <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> la inteligencia artificial es genial <eos>

Input sentence: The computer is on the table.
Traducción: <sos> el ordenador esta sobre la mesa <eos>

Input sentence: She reads a book every day.
Traducción: <sos> ella lee un libro cada dia <eos>

Input sentence: We are students at the university.
Traducción: <so

## Conclusions and Lessons Learned

### Understanding the Transformer Architecture

This activity enabled a practical understanding of the core principles behind the **Transformer architecture**, particularly the central role of the *self-attention* mechanism, which allows each token in a sequence to interact with all others, effectively capturing long-range dependencies.  

Additionally, it clarified the function of the two main components of the model:
- The **encoder**, responsible for representing the meaning of the input sequence.
- The **decoder**, which generates the translation in an autoregressive manner.

This understanding is essential for more advanced applications in Natural Language Processing (NLP).

---

### Insights on the Training Process

Through implementation and experimentation, several key aspects of training deep learning models were observed:

- The **number of training epochs** has a direct impact on learning quality, highlighting that models require multiple passes over the data to capture complex language patterns.
- **Model capacity** and available computational resources significantly influence performance, requiring balanced design decisions between efficiency and accuracy.
- Simple inference strategies such as *greedy decoding* are useful for initial validation, although more advanced methods can further improve results.

These insights reinforce the importance of hyperparameter tuning and iterative experimentation in machine learning workflows.

---

### Importance of Data Preprocessing

This activity also emphasized the critical role of preprocessing in NLP tasks. Decisions such as text normalization, removal of special characters, or vocabulary simplification directly affect:
- Model complexity  
- Training speed  
- Output quality  

This demonstrates that model performance depends not only on architecture but also on how the data is prepared.

---

### Model Evaluation and Validation

It was learned that evaluating translation models should not rely solely on qualitative inspection, but should ideally include formal metrics such as the **BLEU score**, which quantitatively measures similarity between generated translations and reference sentences.

Testing the model on multiple sentences also provided valuable insights into its general behavior and consistency.

---

### Final Reflection

Overall, this activity provided not only hands-on experience in implementing a Transformer model from scratch, but also a deeper understanding of the practical challenges involved in training, evaluating, and interpreting such models.  

Beyond the results obtained, the main value lies in building a solid foundation on how modern NLP models operate, which is essential for developing more advanced and real-world applications.